In [ ]:
import torch
from torchinfo import summary

from llm_from_scratch.pretraining.data.data_loader import create_dataloaders
from llm_from_scratch.pretraining.data.data_loader import load_wikitext103 
from llm_from_scratch.pretraining.model.gpt import GPT

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

## Raw Data

In [ ]:
raw_docs = load_wikitext103()
print(len(raw_docs))
print(raw_docs[1])

## Data Loaders

In [ ]:
dataloader = create_dataloaders()

for batch in dataloader['train']:
    x,y = batch
    print(x.shape, y.shape)
    break

print(dataloader)

## Model

In [ ]:
model = GPT(
    vocab_size=50257,
    n_layers=12,
    d_in=768,
    context_length=1024,
    dropout=0.1,
    n_heads=12,
    qkv_bias=True,
)
sum(p.numel() for p in model.parameters())

In [ ]:
summary(model)

## Loss Calculation

In [ ]:
for batch in dataloader['train']:
    x,y = batch
    print(f"x.shape: {x.shape}, y.shape: {y.shape}")
    break

In [ ]:
x = x.to(device)
y = y.to(device)
model = model.to(device)
logits = model(x)

print(f"y.shape: {y.shape}")
print(f"logits.shape: {logits.shape}")
print(f"y.flatten().shape: {y.flatten().shape}")
print(f"logits.flatten(0,1).shape: {logits.flatten(0,1).shape}")

loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), y.flatten())
print(f"loss: {loss}, type:{type(loss)}")


## Pretrain

In [1]:
from llm_from_scratch.pretraining.model.gpt import GPT
from llm_from_scratch.pretraining.data.data_loader import create_dataloaders
from llm_from_scratch.pretraining.trainer import PreTrainLanguageModelDriver

model = GPT(
    vocab_size=50257,
    n_layers=12,
    d_in=768,
    context_length=1024,
    dropout=0.1,
    n_heads=12,
)

dataloaders = create_dataloaders(batch_size=4, max_length=1024, stride=512)

In [2]:
trainer = PreTrainLanguageModelDriver(model, dataloaders, epochs=3)
trainer.train()

Epoch 0:   1%|          | 530/57871 [09:38<17:23:43,  1.09s/it, loss=18.5365]


KeyboardInterrupt: 